In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.4 MB/s eta 0:00:00


In [3]:
!pip install torch torchvision timm

In [10]:
!pip install onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 116.2 MB/s eta 0:00:00


In [ ]:
%%writefile model_training.py


import os, json, math, time, argparse, random
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm


# Config / CLI

def get_args():
    p = argparse.ArgumentParser()
    p.add_argument("--data_root", type=str, required=True, help="dataset_root with train/ val/ test/")
    p.add_argument("--model", type=str, default="mobilenetv3_small_100")
    p.add_argument("--img_size", type=int, default=224)
    p.add_argument("--batch_size", type=int, default=64)
    p.add_argument("--epochs", type=int, default=100)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--weight_decay", type=float, default=1e-4)
    p.add_argument("--num_workers", type=int, default=4)
    p.add_argument("--out_dir", type=str, default="./outputs")
    p.add_argument("--patience", type=int, default=20, help="early stopping patience")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--mixed_precision", action="store_true", help="use torch autocast")
    p.add_argument("--class_names", type=str, default='["Load Sheet","POD","Pallet","Email","Other"]')
    return p.parse_args()


# Utils

def set_seed(seed):
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def compute_class_weights(train_dir, classes):
    # inverse frequency weights (optional)
    counts = []
    for c in classes:
        cdir = Path(train_dir) / c
        n = sum(1 for _ in cdir.rglob("*") if _.suffix.lower() in [".jpg",".jpeg",".png",".bmp",".tif",".tiff"])
        counts.append(max(n, 1))
    total = sum(counts)
    freqs = [c/total for c in counts]
    inv = [1.0/f for f in freqs]
    s = sum(inv)
    weights = [w/s for w in inv]
    return torch.tensor(weights, dtype=torch.float32)

@torch.no_grad()
def evaluate(model, loader, device, criterion=None):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        if criterion is not None:
            loss_sum += criterion(logits, y).item() * x.size(0)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += y.numel()
    acc = correct / max(total, 1)
    loss = loss_sum / max(total, 1) if criterion is not None else None
    return acc, loss


# Main

def main():
    args = get_args()
    set_seed(args.seed)

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    classes = json.loads(args.class_names)
    assert isinstance(classes, list) and len(classes) == 5, "Expecting 5 classes."
    num_classes = len(classes)

    # Transforms: include strong orientation robustness
    train_tf = transforms.Compose([
        transforms.RandomChoice([
            transforms.Lambda(lambda img: img.rotate(0, expand=True)),
            transforms.Lambda(lambda img: img.rotate(90, expand=True)),
            transforms.Lambda(lambda img: img.rotate(180, expand=True)),
            transforms.Lambda(lambda img: img.rotate(270, expand=True)),
        ]),
        transforms.Resize((args.img_size, args.img_size)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
        transforms.RandomAffine(degrees=3, translate=(0.02,0.02), scale=(0.95,1.05)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((args.img_size, args.img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ])

    # Datasets / Loaders
    train_ds = datasets.ImageFolder(os.path.join(args.data_root, "train"), transform=train_tf)
    val_ds   = datasets.ImageFolder(os.path.join(args.data_root, "val"  ), transform=eval_tf)
    test_ds  = datasets.ImageFolder(os.path.join(args.data_root, "test" ), transform=eval_tf)

    # Make sure folder class order matches our intended CLASSES
    # (If you want to lock class order, pass class_to_idx manually or assert)
    print("Class mapping:", train_ds.class_to_idx)

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True,
                              num_workers=args.num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False,
                              num_workers=args.num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=args.batch_size, shuffle=False,
                              num_workers=args.num_workers, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Model
    model = timm.create_model(args.model, pretrained=True, num_classes=num_classes)
    model.to(device)

    # Loss with optional class weights (helps for imbalance)
    class_weights = compute_class_weights(os.path.join(args.data_root, "train"), classes).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    scaler = torch.cuda.amp.GradScaler(enabled=args.mixed_precision)

    best_val_acc = 0.0
    best_path = out_dir / "best_mnv3.pt"
    patience = args.patience
    patience_ct = 0

    for epoch in range(1, args.epochs + 1):
        model.train()
        t0 = time.time()
        total, running_loss = 0, 0.0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=args.mixed_precision):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * x.size(0)
            total += y.numel()

        scheduler.step()
        train_loss = running_loss / max(total, 1)
        val_acc, val_loss = evaluate(model, val_loader, device, criterion)

        dt = time.time() - t0
        print(f"Epoch {epoch:03d}/{args.epochs} | "
              f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} | {dt:.1f}s")

        # Early stopping on val_acc
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_ct = 0
            torch.save({"model_state": model.state_dict(),
                        "classes": classes,
                        "class_to_idx": train_ds.class_to_idx}, best_path)
            print(f"Saved best to {best_path} (val_acc={val_acc:.4f})")
        else:
            patience_ct += 1
            if patience_ct >= patience:
                print("Early stopping.")
                break

    # Final eval on test
    checkpoint = torch.load(best_path, map_location="cpu")
    model.load_state_dict(checkpoint["model_state"])
    model.to(device).eval()
    test_acc, _ = evaluate(model, test_loader, device, criterion=None)
    print(f"Test accuracy: {test_acc:.4f}")

    # Export ONNX
    onnx_path = out_dir / "model.onnx"
    model.eval()
    dummy = torch.randn(1, 3, args.img_size, args.img_size, device=device)
    torch.onnx.export(
        model, dummy, str(onnx_path),
        input_names=["input"], output_names=["logits"],
        opset_version=13, dynamic_axes={"input": {0: "batch"}}
    )
    print(f"Exported ONNX to {onnx_path}")

    # Save class list and normalization used
    meta = {
        "classes": classes,
        "img_size": args.img_size,
        "mean": [0.485, 0.456, 0.406],
        "std":  [0.229, 0.224, 0.225],
        "class_to_idx": checkpoint["class_to_idx"]
    }
    (out_dir / "meta.json").write_text(json.dumps(meta, indent=2))
    print(f"Wrote {out_dir / 'meta.json'}")

if __name__ == "__main__":
    main()

In [ ]:
!python /content/model_training.py --data_root /content/drive/MyDrive/doc_data --epochs 100 --mixed_precision

In [ ]:
import torch
import timm
from pathlib import Path
from PIL import Image
from torchvision import transforms
import os


# Load Model Function

def load_model(pt_path, device="cpu"):
    # Load checkpoint
    checkpoint = torch.load(pt_path, map_location=device)

    # Load classes + class_to_idx
    classes = checkpoint["classes"]   # e.g. ["Email", "Invoice", "Receipt"]
    class_to_idx = checkpoint["class_to_idx"]  # e.g. {"Email":0, "Invoice":1, "Receipt":2}
    num_classes = len(classes)

    # Recreate model
    model = timm.create_model("mobilenetv3_small_100", pretrained=False, num_classes=num_classes)
    model.load_state_dict(checkpoint["model_state"])
    model.to(device).eval()

    return model, classes, class_to_idx



# Preprocess Function

def preprocess_image(img_path, img_size=224):
    tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406),
                             std=(0.229, 0.224, 0.225)),
    ])
    img = Image.open(img_path).convert("RGB")
    return tf(img).unsqueeze(0)  # Add batch dimension



# Prediction Function

def predict_image(model, img_path, classes, class_to_idx, device="cpu", img_size=224):
    x = preprocess_image(img_path, img_size).to(device)
    with torch.no_grad():
        logits = model(x)
        probs = torch.nn.functional.softmax(logits, dim=1)  # convert to probabilities
        conf, pred_idx = torch.max(probs, 1)  # best class + confidence

    #  Ensure correct mapping: idx → class
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    pred_class = idx_to_class[pred_idx.item()]

    return pred_class, pred_idx.item(), conf.item()



# Run Inference

if __name__ == "__main__":
    # ---- Update these paths ----
    pt_path = r"/content/outputs/best_mnv3.pt"
    img_dir = r"/content/drive/MyDrive/doc_data/test/Pallet"
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load model
    model, classes, class_to_idx = load_model(pt_path, device)

    # Loop through all files in the directory
    for root, _, files in os.walk(img_dir):
        for file in files:
            if file.lower().endswith((".jpg", ".jpeg", ".png")):  # only image files
                img_path = os.path.join(root, file)

                try:
                    pred_class, pred_idx, confidence = predict_image(model, img_path, classes, class_to_idx, device)
                    print(f"Prediction for {file}: {pred_class} (index={pred_idx}, confidence={confidence:.4f})")
                except Exception as e:
                    print(f"Error processing {file}: {e}")